### Импорты

In [1]:
import pandas as pd
import requests
from io import StringIO
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### Сбор данных

**kepid**, **kepoi_name**, **kepler_name** - id/имена космических объектов. Возможно, будут служить ключами для последующего соединения с данными о кривых блеска соответствующих звёзд

**koi_pdisposition** и **koi_score** - рещультаты предсказания модели nasa - полезно посмотреть, нельзя передавать в модель во избежание data leakage

**koi_disposition** - целевая переменная, нужно научиться предсказать

**Остальные параметры** - метрики для обучения модели

In [2]:
columns = [
    "kepid", "kepoi_name", "kepler_name",
    "koi_disposition", "koi_pdisposition", "koi_score",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_period", "koi_duration", "koi_depth", "koi_prad",
    "koi_teq", "koi_insol", "koi_model_snr",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag",
]
query = f"select {','.join(columns)} from cumulative"

response = requests.get(
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync",
    params={"query": query, "format": "csv"}
)
response.raise_for_status()

koi = pd.read_csv(StringIO(response.text))
koi.to_csv("../data/processed_physical_signs/koi_cumulative.csv", index=False)

In [3]:
koi_binary = koi[koi["koi_disposition"] != "CANDIDATE"].drop(columns=["koi_pdisposition", "kepid", "kepoi_name", "kepler_name"]).copy()
koi_binary["koi_disposition"] = koi_binary["koi_disposition"].map({"CONFIRMED": 1, "FALSE POSITIVE": 0})
corr_matrix = koi_binary.corr()
display(corr_matrix["koi_disposition"].sort_values(ascending=False))

koi_binary_zero = koi_binary[koi_binary["koi_disposition"] == 0].copy()
koi_binary_one = koi_binary[koi_binary["koi_disposition"] == 1].copy()
display(koi_binary_one)
display(koi_binary_zero)

one_with_flags = koi_binary["koi_disposition"][(koi_binary["koi_disposition"] == 1) & ((koi_binary["koi_fpflag_nt"] + koi_binary["koi_fpflag_ss"] + koi_binary["koi_fpflag_co"] + koi_binary["koi_fpflag_ec"]) > 0)].count()
print(one_with_flags)
zero_without_flags = koi_binary["koi_disposition"][(koi_binary["koi_disposition"] == 0) & ((koi_binary["koi_fpflag_nt"] + koi_binary["koi_fpflag_ss"] + koi_binary["koi_fpflag_co"] + koi_binary["koi_fpflag_ec"]) == 0)].count()
print(zero_without_flags)

koi_disposition    1.000000
koi_score          0.946880
koi_slogg          0.181666
koi_kepmag         0.056550
koi_fpflag_nt     -0.013029
koi_prad          -0.024657
koi_insol         -0.039214
koi_srad          -0.085304
koi_period        -0.160019
koi_duration      -0.161904
koi_model_snr     -0.210773
koi_steff         -0.222013
koi_depth         -0.241792
koi_teq           -0.277274
koi_fpflag_ec     -0.318194
koi_fpflag_co     -0.433896
koi_fpflag_ss     -0.476848
Name: koi_disposition, dtype: float64

,koi_disposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,1,1.000,0,0,0,0,9.488036,2.9575,615.8,2.26,793.0,93.59,35.8,5455.0,4.467,0.927,15.347
1,1,0.969,0,0,0,0,54.418383,4.5070,874.8,2.83,443.0,9.11,25.8,5455.0,4.467,0.927,15.347
4,1,1.000,0,0,0,0,2.525592,1.6545,603.3,2.75,1406.0,926.16,40.9,6031.0,4.438,1.046,15.509
5,1,1.000,0,0,0,0,11.094321,4.5945,1517.5,3.90,835.0,114.81,66.5,6046.0,4.486,0.972,15.714
6,1,1.000,0,0,0,0,4.134435,3.1402,686.0,2.77,1160.0,427.65,40.2,6046.0,4.486,0.972,15.714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8817,1,1.000,0,0,0,0,4.485592,0.8161,2265.0,0.95,365.0,4.20,15.0,3236.0,5.097,0.193,15.737
8956,1,NaN,0,0,0,0,8.152759,1.7460,16536.0,2.44,305.0,2.03,9.2,3327.0,5.113,0.189,17.475
9014,1,0.771,0,0,0,0,384.847556,9.9690,189.9,1.09,220.0,0.56,12.3,5579.0,4.580,0.798,13.426
9083,1,0.000,0,0,0,0,3.875943,2.3140,58.6,0.68,1081.0,323.21,14.2,5713.0,4.541,0.893,12.750


,koi_disposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
3,0,0.000,0,1,0,0,1.736952,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,4.564,0.791,15.597
8,0,0.000,0,1,1,0,7.361790,5.02200,233.7,39.21,1342.0,767.22,47.7,6227.0,3.986,1.958,12.660
14,0,0.000,0,1,0,0,11.521446,3.63990,17984.3,150.51,753.0,75.88,622.1,5795.0,4.554,0.848,15.472
15,0,0.000,0,1,0,0,19.403938,12.21550,8918.7,7.18,523.0,17.69,214.7,5043.0,4.591,0.680,15.487
16,0,0.000,0,1,0,0,19.221389,4.79843,74284.0,49.29,698.0,55.97,2317.0,6117.0,4.496,0.947,15.341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9557,0,0.008,1,0,0,0,373.893980,27.66000,730.0,2.51,206.0,0.42,18.5,5263.0,4.574,0.699,14.911
9558,0,0.000,0,0,0,1,8.589871,4.80600,87.7,1.11,929.0,176.40,8.4,5638.0,4.296,1.088,14.478
9559,0,0.000,0,1,1,0,0.527699,3.22210,1579.2,29.35,2088.0,4500.53,453.3,5638.0,4.529,0.903,14.082
9561,0,0.021,0,0,1,0,0.681402,0.86500,103.6,1.07,2218.0,5713.41,12.3,6173.0,4.447,1.041,15.385


18
95


### Опасность
*Из 2746 строк с положительным koi_disposition всего 17 строк имеют хоть какой-то из флагов, а из 4581 с отрицательным koi_disposition - всего 48 не имеют флагов вообще. С учётом того, что при первом запуске моделей на датасете с этими флагами (см. ноутбук ниже) результаты были подозрительно хороши (XGB с recall 99.4% на кросс-валидации), можно предположить, что наличие любого флага слишком сильно коррелирует с правильным результатом koi_disposition, и лучше избавиться от флагов совсем, чтобы сделать результаты более справедливыми*

In [4]:
koi.drop(columns=["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"], inplace=True)

### Структура скачанных данных

In [5]:
print(koi.shape)
print(koi["koi_disposition"].value_counts())
koi.head(10)

(9564, 17)
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64


,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,9.488036,2.95750,615.8,2.26,793.0,93.59,35.8,5455.0,4.467,0.927,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,54.418383,4.50700,874.8,2.83,443.0,9.11,25.8,5455.0,4.467,0.927,15.347
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,19.899140,1.78220,10829.0,14.60,638.0,39.30,76.3,5853.0,4.544,0.868,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,1.736952,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,4.564,0.791,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,2.525592,1.65450,603.3,2.75,1406.0,926.16,40.9,6031.0,4.438,1.046,15.509
5,10872983,K00756.01,Kepler-228 d,CONFIRMED,CANDIDATE,1.000,11.094321,4.59450,1517.5,3.90,835.0,114.81,66.5,6046.0,4.486,0.972,15.714
6,10872983,K00756.02,Kepler-228 c,CONFIRMED,CANDIDATE,1.000,4.134435,3.14020,686.0,2.77,1160.0,427.65,40.2,6046.0,4.486,0.972,15.714
7,10872983,K00756.03,Kepler-228 b,CONFIRMED,CANDIDATE,0.992,2.566589,2.42900,226.5,1.59,1360.0,807.74,15.0,6046.0,4.486,0.972,15.714
8,6721123,K00114.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,7.361790,5.02200,233.7,39.21,1342.0,767.22,47.7,6227.0,3.986,1.958,12.660
9,10910878,K00757.01,Kepler-229 c,CONFIRMED,CANDIDATE,1.000,16.068647,3.53470,4914.3,5.76,600.0,30.75,161.9,5031.0,4.485,0.848,15.841


In [6]:
koi.describe()

,kepid,koi_score,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
count,9.564000e+03,8054.000000,9564.000000,9564.000000,9.201000e+03,9201.000000,9201.000000,9.243000e+03,9201.000000,9201.000000,9201.000000,9201.000000,9563.000000
mean,7.690628e+06,0.480829,75.671358,5.621606,2.379134e+04,102.891778,1085.385828,7.745737e+03,259.895001,5706.823280,4.310157,1.728712,14.264606
std,2.653459e+06,0.476928,1334.744046,6.471554,8.224268e+04,3077.639126,856.351161,1.592047e+05,795.806615,796.857947,0.432606,6.127185,1.385448
min,7.574500e+05,0.000000,0.241843,0.052000,0.000000e+00,0.080000,25.000000,0.000000e+00,0.000000,2661.000000,0.047000,0.109000,6.966000
25%,5.556034e+06,0.000000,2.733684,2.437750,1.599000e+02,1.400000,539.000000,2.015000e+01,12.000000,5310.000000,4.218000,0.829000,13.440000
50%,7.906892e+06,0.334000,9.752831,3.792600,4.211000e+02,2.390000,878.000000,1.416000e+02,23.000000,5767.000000,4.438000,1.000000,14.520000
75%,9.873066e+06,0.998000,40.715178,6.276500,1.473400e+03,14.930000,1379.000000,8.702900e+02,78.000000,6112.000000,4.543000,1.345000,15.322000
max,1.293514e+07,1.000000,129995.778400,138.540000,1.541400e+06,200346.000000,14667.000000,1.094755e+07,9054.700000,15896.000000,5.364000,229.908000,20.003000


In [7]:
koi.columns

Index(['kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
       'koi_pdisposition', 'koi_score', 'koi_period', 'koi_duration',
       'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr',
       'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag'],
      dtype='object')

In [8]:
koi[["koi_disposition", "koi_pdisposition", "koi_score", "kepoi_name", "kepid", "kepler_name"]].head()

,koi_disposition,koi_pdisposition,koi_score,kepoi_name,kepid,kepler_name
0,CONFIRMED,CANDIDATE,1.000,K00752.01,10797460,Kepler-227 b
1,CONFIRMED,CANDIDATE,0.969,K00752.02,10797460,Kepler-227 c
2,CANDIDATE,CANDIDATE,0.000,K00753.01,10811496,NaN
3,FALSE POSITIVE,FALSE POSITIVE,0.000,K00754.01,10848459,NaN
4,CONFIRMED,CANDIDATE,1.000,K00755.01,10854555,Kepler-664 b


### Просмотр пропусков

Пропуски в kepler_name можно не учитывать - это естественно, что их так много, т. к. имена дают только подтверждённым космическим объектам. Так как этот параметр не пойдёт в модель, можно оставить как есть

koi_score - аналогично: просто уберётся столбец до передачи данных модели, чтобы она не подсматривала результаты модели nasa

Остальные пропуски - 363 пропуска в почти что половине колонок. Если это всё одни и те же строки, можно спокойно удалить. Проверим это, убрав пропуски по одному из этих параметров и посмотреть isna заново: если пропусков больше нет - значит, гипотеза была верна

In [9]:
koi.isna().sum()

kepid                  0
kepoi_name             0
kepler_name         6816
koi_disposition        0
koi_pdisposition       0
koi_score           1510
koi_period             0
koi_duration           0
koi_depth            363
koi_prad             363
koi_teq              363
koi_insol            321
koi_model_snr        363
koi_steff            363
koi_slogg            363
koi_srad             363
koi_kepmag             1
dtype: int64

Просто было интересно посмотреть

In [10]:
koi[koi["koi_score"].isna() == True]

,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
24,2306756,K00113.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,386.603053,6.8057,24926.6,41.50,297.0,1.84,994.0,5543.0,4.081,1.580,12.394
75,12459725,K00789.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,14.179819,2.5050,459.9,1.88,711.0,60.35,11.2,5799.0,4.539,0.878,15.727
99,3832474,K00806.03,Kepler-30 b,CONFIRMED,CANDIDATE,NaN,29.159862,4.8030,488.0,1.91,524.0,17.85,11.6,5485.0,4.556,0.867,15.403
123,5115978,K00823.01,NaN,CANDIDATE,CANDIDATE,NaN,1.028437,1.8606,8098.0,72.77,1947.0,3400.48,89.6,6228.0,4.428,0.990,15.202
318,10460629,K01168.01,NaN,CANDIDATE,CANDIDATE,NaN,856.672115,23.1440,858.1,3.87,242.0,0.81,39.2,6449.0,4.232,1.309,13.997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9155,9025662,K07599.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,293.706960,11.0300,1575.0,6140.44,274.0,1.34,11.2,6054.0,4.508,0.893,15.510
9162,5652983,K00371.02,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,9.058466,6.3760,61.8,2.46,1310.0,695.41,17.9,5198.0,3.614,3.207,12.193
9163,9092897,K07600.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,3.916915,2.1100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.940
9165,9292130,K07602.01,NaN,FALSE POSITIVE,FALSE POSITIVE,NaN,362.747593,16.4500,132.4,2.51,384.0,5.13,13.0,6176.0,3.885,2.160,10.662


In [11]:
koi

,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,9.488036,2.95750,615.8,2.26,793.0,93.59,35.8,5455.0,4.467,0.927,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,54.418383,4.50700,874.8,2.83,443.0,9.11,25.8,5455.0,4.467,0.927,15.347
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,19.899140,1.78220,10829.0,14.60,638.0,39.30,76.3,5853.0,4.544,0.868,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,1.736952,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,4.564,0.791,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,2.525592,1.65450,603.3,2.75,1406.0,926.16,40.9,6031.0,4.438,1.046,15.509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,10090151,K07985.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0.527699,3.22210,1579.2,29.35,2088.0,4500.53,453.3,5638.0,4.529,0.903,14.082
9560,10128825,K07986.01,NaN,CANDIDATE,CANDIDATE,0.497,1.739849,3.11400,48.5,0.72,1608.0,1585.81,10.6,6119.0,4.444,1.031,14.757
9561,10147276,K07987.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.021,0.681402,0.86500,103.6,1.07,2218.0,5713.41,12.3,6173.0,4.447,1.041,15.385
9562,10155286,K07988.01,NaN,CANDIDATE,CANDIDATE,0.092,333.486169,3.19900,639.1,19.30,557.0,22.68,14.0,4989.0,2.992,7.824,10.998


In [12]:
koi.drop(columns=["koi_score", "koi_pdisposition"], inplace=True)
koi.dropna(subset=["koi_depth", "koi_kepmag"], inplace=True)

Всё отлично, пропусков больше нет

In [13]:
koi.isna().sum()

kepid                 0
kepoi_name            0
kepler_name        6454
koi_disposition       0
koi_period            0
koi_duration          0
koi_depth             0
koi_prad              0
koi_teq               0
koi_insol             0
koi_model_snr         0
koi_steff             0
koi_slogg             0
koi_srad              0
koi_kepmag            0
dtype: int64

Так как это данные о разных звёздах с официального сайта nasa, дубликатов быть не должно, но всё равно можно проверить

In [14]:
display(koi.shape)
koi.drop_duplicates(inplace=True)   # Только полные дубликаты
koi.shape

(9200, 15)

(9200, 15)

### Стандартизация данных

Делаем масштабирование только по числовым признакам

In [15]:
col_for_scaling = koi.drop(columns=["kepid", "kepoi_name", "kepler_name", 'koi_disposition']).columns

In [16]:
col_for_scaling

Index(['koi_period', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq',
       'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_slogg', 'koi_srad',
       'koi_kepmag'],
      dtype='object')

Смотрим, насколько существенные имеются выбросы в этих признаках, так как стандартизация чувствительна к сильным выбросам, а датасет основан на информации о разных звёздах с абсолютно разными показателями, которые могут отличаться кардинально.

Если skew будет показывать для признака число, значительно отличное от 1 - до стандартизации лучше применить логарифмирование

In [17]:
print(koi[col_for_scaling].skew().sort_values(ascending=False))

koi_period       94.668430
koi_prad         52.116120
koi_insol        49.852434
koi_srad         20.945969
koi_duration      5.872071
koi_model_snr     5.306784
koi_depth         5.260440
koi_teq           3.505657
koi_steff         0.749772
koi_kepmag       -0.809560
koi_slogg        -3.432862
dtype: float64


In [18]:
log_cols = ["koi_period", "koi_prad", "koi_insol", "koi_srad", "koi_duration", "koi_model_snr", "koi_depth", "koi_teq", "koi_slogg"]

In [19]:
koi["koi_disposition"].value_counts()

koi_disposition
FALSE POSITIVE    4581
CONFIRMED         2746
CANDIDATE         1873
Name: count, dtype: int64

Разделяем данные: так как значение CANDIDATE метрики koi_disposition, по сути, несёт в себе только отсутствие решения модели nasa, это достаточно сильно может запутать будущую модель, заставив её не определять, имеет ли звезда экзопланету, а просто подгонять значения под выводф модели nasa. Стоит убрать строки с этим значением, оставив для модели задачу бинарной классификации (CONFIRMED/FALSE POSITIVE), но строки с CANDIDATE оставим отдельно для возможного будущего изучения

In [20]:
koi_candidates = koi[koi["koi_disposition"] == "CANDIDATE"].drop(columns=["kepid", "kepoi_name", "kepler_name"]).copy()
koi_binary = koi[koi["koi_disposition"] != "CANDIDATE"].drop(columns=["kepid", "kepoi_name", "kepler_name"]).copy()
keys = koi[["kepid", "kepoi_name", "kepler_name"]].copy()  # сохраним все метрики-идентификаторы тут

In [21]:
display(koi_candidates["koi_disposition"].value_counts())
display(koi_binary["koi_disposition"].value_counts())
keys.head()

koi_disposition
CANDIDATE    1873
Name: count, dtype: int64

koi_disposition
FALSE POSITIVE    4581
CONFIRMED         2746
Name: count, dtype: int64

,kepid,kepoi_name,kepler_name
0,10797460,K00752.01,Kepler-227 b
1,10797460,K00752.02,Kepler-227 c
2,10811496,K00753.01,NaN
3,10848459,K00754.01,NaN
4,10854555,K00755.01,Kepler-664 b


**train/test**

In [22]:
X = koi_binary.drop(columns=["koi_disposition"]).copy()
y = koi_binary["koi_disposition"].copy()
y = y.map({"CONFIRMED": 1, "FALSE POSITIVE": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=52)

**Логарифмизация**

In [23]:
for col in log_cols:
    if ((X_train[col] >= 0).all()):
        X_train[col] = np.log1p(X_train[col])
        print(f"{col} прошёл логарифмизацию")

koi_period прошёл логарифмизацию
koi_prad прошёл логарифмизацию
koi_insol прошёл логарифмизацию
koi_srad прошёл логарифмизацию
koi_duration прошёл логарифмизацию
koi_model_snr прошёл логарифмизацию
koi_depth прошёл логарифмизацию
koi_teq прошёл логарифмизацию
koi_slogg прошёл логарифмизацию


In [24]:
for col in log_cols:
    if ((X_test[col] >= 0).all()):
        X_test[col] = np.log1p(X_test[col])
        print(f"{col} прошёл логарифмизацию")

koi_period прошёл логарифмизацию
koi_prad прошёл логарифмизацию
koi_insol прошёл логарифмизацию
koi_srad прошёл логарифмизацию
koi_duration прошёл логарифмизацию
koi_model_snr прошёл логарифмизацию
koi_depth прошёл логарифмизацию
koi_teq прошёл логарифмизацию
koi_slogg прошёл логарифмизацию


**Стандартизация**

In [25]:
for col in col_for_scaling:
    X_train[col] = (X_train[col] - X_train[col].mean())/X_train[col].std()

In [26]:
for col in col_for_scaling:
    X_test[col] = (X_test[col] - X_train[col].mean())/X_train[col].std()

### Обучение моделей

### Линейная регрессия

In [27]:
log_regression = LogisticRegression(
    class_weight="balanced",
    random_state=52
)
log_regression.fit(X_train, y_train)

y_pred = log_regression.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[  0 933]
 [  0 533]]
              precision    recall  f1-score   support

           0      0.000     0.000     0.000       933
           1      0.364     1.000     0.533       533

    accuracy                          0.364      1466
   macro avg      0.182     0.500     0.267      1466
weighted avg      0.132     0.364     0.194      1466



c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

### Случайный лес

In [28]:
rand_forest = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=52
)
rand_forest.fit(X_train, y_train)

y_pred = rand_forest.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[933   0]
 [533   0]]
              precision    recall  f1-score   support

           0      0.636     1.000     0.778       933
           1      0.000     0.000     0.000       533

    accuracy                          0.636      1466
   macro avg      0.318     0.500     0.389      1466
weighted avg      0.405     0.636     0.495      1466



c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

### Градиентный бустинг

In [29]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=52
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[933   0]
 [533   0]]
              precision    recall  f1-score   support

           0      0.636     1.000     0.778       933
           1      0.000     0.000     0.000       533

    accuracy                          0.636      1466
   macro avg      0.318     0.500     0.389      1466
weighted avg      0.405     0.636     0.495      1466



c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vladislav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

На данный момент лучший результат в recall по угадыванию именно позитивных случаев побеждает XGB. Следует убедиться при помощи валидаии

### Кросс-валидация

In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_regression_valid = LogisticRegression(class_weight="balanced", max_iter=1000)

scores = cross_val_score(log_regression_valid, X_train, y_train, cv=cv, scoring="recall")
print(scores, scores.mean())

[0.89841986 0.90723982 0.90950226 0.89390519 0.88487585] 0.8987885968765001


In [31]:
rand_forest_valid = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=52
)

scores = cross_val_score(rand_forest_valid, X_train, y_train, cv=cv, scoring="recall")
print(scores, scores.mean())

[0.89390519 0.92307692 0.90497738 0.91196388 0.89841986] 0.9064686475388907


In [32]:
xgb_model_valid = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=52
)

scores = cross_val_score(xgb_model_valid, X_train, y_train, cv=cv, scoring="recall")
print(scores, scores.mean())

[0.9255079  0.93891403 0.92533937 0.93002257 0.9255079 ] 0.9290583536765983


### Вывод
В обоих случаях побеждает XGB, причём с очень хорошими результатами (92.9% на валидаии), похтому следует выбрать его в качестве основной модели для этой задачи